<a href="https://colab.research.google.com/github/kpal002/Jupyter-notebooks-for-Blogs/blob/main/transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer from Scratch
**Tokenized Universe — Post 4**

Building every component of a modern Transformer LM from scratch, following CS336 Assignment 1 constraints:
- No `nn.Linear`, `nn.Embedding`, or `nn.MultiheadAttention`
- All weights initialized manually
- All operations implemented using raw PyTorch

Components covered:
1. `Linear` — weight matrix with Glorot init
2. `Embedding` — token lookup table
3. `RMSNorm` — root mean square layer normalization
4. `SwiGLUFFN` — gated feed-forward network
5. `softmax` — numerically stable softmax
6. `scaled_dot_product_attention` — the core attention operation
7. `RotaryPositionalEmbedding` — RoPE
8. `CausalMultiHeadSelfAttention` — full multi-head attention with causal mask
9. `TransformerBlock` — one layer with pre-norm + residual connections
10. `TransformerLM` — the full language model


In [1]:
# Standard imports
import math
import torch
import torch.nn as nn


## 1. Linear Layer

A weight matrix `W` of shape `(out, in)` initialized with Glorot (Xavier) uniform — sigma = sqrt(2 / (fan_in + fan_out)), truncated at ±3σ.

Forward: `y = x @ W.T` — implemented via `einsum` so it handles arbitrary batch dimensions cleanly.

In [2]:
class Linear(nn.Module):
    """
    A linear (fully-connected) layer without bias.

    Weight shape: (out_features, in_features) — same convention as nn.Linear.
    Initialized with Glorot (Xavier) normal, truncated at ±3σ.

    Args:
        in_features  (int): Size of each input sample.
        out_features (int): Size of each output sample.
        device: Target device (e.g. 'cuda', 'cpu').
        dtype:  Target dtype (e.g. torch.float32, torch.bfloat16).
    """

    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()

        # Allocate the weight parameter (un-initialized)
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features, device=device, dtype=dtype)
        )

        # Glorot (Xavier) normal: sigma = sqrt(2 / (fan_in + fan_out))
        # Truncated at ±3σ to avoid extreme outliers at init
        sigma = math.sqrt(2.0 / (in_features + out_features))
        nn.init.trunc_normal_(self.weight, mean=0.0, std=sigma, a=-3 * sigma, b=3 * sigma)

    def forward(self, x):
        """
        Compute y = x @ W^T.

        Uses einsum so any number of leading batch dimensions work
        without explicit reshaping.

        Args:
            x: (..., in_features)
        Returns:
            y: (..., out_features)
        """
        # '... d_in, d_out d_in -> ... d_out'  reads as:
        #   for every position in batch (...), dot x's d_in with W's d_in,
        #   producing one scalar per d_out
        return torch.einsum("...i,oi->...o", x, self.weight)


## 2. Embedding Layer

A lookup table: shape `(vocab_size, d_model)`. Given integer token IDs, return the corresponding row vectors.

Initialized with truncated normal (mean=0, std=1).

In [3]:
class Embedding(nn.Module):
    """
    A simple token embedding lookup table.

    Stores a matrix of shape (num_embeddings, embedding_dim).
    Given a tensor of integer token IDs, returns the corresponding rows.

    Args:
        num_embeddings (int): Vocabulary size.
        embedding_dim  (int): Dimension of each embedding vector.
        device: Target device.
        dtype:  Target dtype.
    """

    def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None):
        super().__init__()

        # The embedding table: one row per vocabulary token
        self.weight = nn.Parameter(
            torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype)
        )

        # Truncated normal init — keeps vectors near the unit sphere
        nn.init.trunc_normal_(self.weight, mean=0.0, std=1.0, a=-3.0, b=3.0)

    def forward(self, token_ids):
        """
        Look up embeddings for a batch of token IDs.

        Args:
            token_ids: Integer tensor of any shape (...)
        Returns:
            Embedding vectors of shape (..., embedding_dim)
        """
        # Direct index into the weight matrix — equivalent to one-hot @ weight
        return self.weight[token_ids]


## 3. RMSNorm

Root Mean Square Layer Normalization. Normalizes by the RMS of the activations (no mean centering), then scales by a learned vector `g`.

Computed in float32 for numerical stability, then cast back to the input dtype.

In [4]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization (RMSNorm).

    Unlike LayerNorm, RMSNorm skips mean subtraction and normalizes
    only by the root-mean-square of the activations, then applies
    a learned per-dimension scale.

    Args:
        d_model (int):   Model dimension (last axis of input).
        eps     (float): Small constant for numerical stability.
        device: Target device.
        dtype:  Target dtype.
    """

    def __init__(self, d_model, eps=1e-5, device=None, dtype=None):
        super().__init__()
        self.eps = eps
        # Learned scale vector, initialized to ones (identity at init)
        self.weight = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype))

    def forward(self, x):
        """
        Normalize x by its RMS and scale by the learned weight.

        Args:
            x: (..., d_model)
        Returns:
            Normalized tensor of same shape as x.
        """
        # Upcast to float32 to avoid precision loss during the norm computation
        in_dtype = x.dtype
        x = x.to(torch.float32)

        # RMS = sqrt( mean(x^2) + eps )
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

        # Normalize then apply the learned scale
        result = (x / rms) * self.weight.to(torch.float32)

        # Cast back to original dtype (e.g. bfloat16 in mixed-precision training)
        return result.to(in_dtype)


## 4. SwiGLU Feed-Forward Network

SwiGLU = SiLU activation (sigmoid-weighted linear unit) + a gating mechanism.

Three projections: `W1` (gate input), `W3` (values), `W2` (output).
The hidden dim defaults to `ceil((8/3 * d_model) / 64) * 64` — parameter-equivalent to a 4× FFN but rounded up to the nearest multiple of 64 for GPU alignment.

In [5]:
class SwiGLUFFN(nn.Module):
    """
    SwiGLU Feed-Forward Network.

    SwiGLU combines a SiLU gate with an element-wise gating mechanism:
        output = W2( SiLU(W1(x)) * W3(x) )

    Three weight matrices:
      W1: projects x to the gate branch (gate_input)
      W3: projects x to the value branch (what to pass through)
      W2: projects the gated hidden state back to d_model

    The default hidden dimension is chosen to be parameter-equivalent
    to a standard 2-matrix FFN with d_ff = 4 * d_model, then rounded
    up to the nearest multiple of 64 for GPU memory alignment.

    Args:
        d_model (int): Model dimension.
        d_ff    (int): Hidden dimension. Auto-computed if None.
        device: Target device.
        dtype:  Target dtype.
    """

    def __init__(self, d_model, d_ff=None, device=None, dtype=None):
        super().__init__()

        if d_ff is None:
            # (8/3) * d_model is parameter-equivalent to 4*d_model with 3 matrices instead of 2
            # Round up to nearest multiple of 64 for efficient GPU tensor alignment
            d_ff = math.ceil(int(8 / 3 * d_model) / 64) * 64

        # Gate branch: produces the activation that controls information flow
        self.W1 = Linear(d_model, d_ff, device=device, dtype=dtype)

        # Output projection: maps gated hidden state back to d_model
        self.W2 = Linear(d_ff, d_model, device=device, dtype=dtype)

        # Value branch: produces the content that gets gated
        self.W3 = Linear(d_model, d_ff, device=device, dtype=dtype)

    def forward(self, x):
        """
        Apply SwiGLU transformation.

        Args:
            x: (..., d_model)
        Returns:
            Tensor of shape (..., d_model)
        """
        # Step 1: Compute gate — SiLU(W1(x)) = x * sigmoid(x)
        gate_input = self.W1(x)
        gate = gate_input * torch.sigmoid(gate_input)  # SiLU activation

        # Step 2: Compute values — what information the gate will filter
        values = self.W3(x)

        # Step 3: Element-wise gating, then project back to d_model
        return self.W2(gate * values)


## 5. Numerically Stable Softmax

Subtracts the row maximum before exponentiation to prevent `exp(large number)` overflow. Mathematically equivalent to standard softmax (the constant cancels in numerator and denominator).

In [6]:
def softmax(x, dim):
    """
    Numerically stable softmax along a given dimension.

    Subtracts the max value before exponentiating to avoid overflow
    from large attention scores. The shift cancels out mathematically:
        softmax(x) == softmax(x - max(x))

    Args:
        x   (Tensor): Input tensor.
        dim (int):    Dimension along which to apply softmax.

    Returns:
        Tensor of same shape as x, values sum to 1 along `dim`.
    """
    # Subtract max for numerical stability (prevents exp() overflow)
    x_shifted = x - x.max(dim=dim, keepdim=True).values

    # Standard softmax: exp(x) / sum(exp(x))
    exp_x = torch.exp(x_shifted)
    return exp_x / exp_x.sum(dim=dim, keepdim=True)


## 6. Scaled Dot-Product Attention

The core attention operation:
1. Compute raw scores: `Q @ K^T / sqrt(d_k)`
2. Apply causal mask (set future positions to -inf)
3. Softmax over the key dimension
4. Weighted sum of values: `attn_weights @ V`

In [7]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Scaled dot-product attention.

    Computes:
        Attention(Q, K, V) = softmax( QK^T / sqrt(d_k) ) * V

    Args:
        query: (..., n, d_k)  — query vectors
        key:   (..., m, d_k)  — key vectors
        value: (..., m, d_v)  — value vectors
        mask:  (..., n, m) boolean tensor, True = attend, False = mask out

    Returns:
        Output tensor of shape (..., n, d_v)
    """
    # d_k is the key/query dimension — used for scaling
    d_k = query.shape[-1]

    # Step 1: Compute raw attention scores — QK^T
    # '... n d_k, ... m d_k -> ... n m' = for each query position n,
    # compute dot product with every key position m
    scores = torch.einsum("...nd,...md->...nm", query, key)

    # Step 2: Scale by 1/sqrt(d_k) to prevent softmax saturation
    scores = scores / math.sqrt(d_k)

    # Step 3: Apply mask — positions where mask is False get -inf,
    # so after softmax they become 0 (no attention)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))

    # Step 4: Softmax over key dimension → attention weights
    attn_weights = softmax(scores, dim=-1)

    # Step 5: Weighted sum over value vectors
    # '... n m, ... m d_v -> ... n d_v'
    return torch.einsum("...nm,...mv->...nv", attn_weights, value)


## 7. Rotary Positional Embedding (RoPE)

Instead of adding positional encodings, RoPE *rotates* query and key vectors by an angle proportional to their position. This encodes relative distances directly in the dot product — closer tokens produce higher scores naturally.

Applied only to Q and K, not V — position tells you *where to look*, not *what to say*.

In [8]:
class RotaryPositionalEmbedding(nn.Module):
    """
    Rotary Positional Embedding (RoPE).

    Rotates query/key vectors in 2D sub-spaces using position-dependent angles.
    The rotation angle for dimension pair (2i, 2i+1) at position p is:
        theta_i = p / (base ** (2i / d_k))

    Precomputed cos/sin tables are stored as non-persistent buffers
    (not saved in model checkpoints, recomputed on load).

    RoPE is applied to Q and K only — NOT to V.
    Position encodes where to look, not what to say.

    Args:
        theta       (float): Base frequency (10000 in original RoPE, 500000 in LLaMA 3).
        d_k         (int):   Head dimension (must be even).
        max_seq_len (int):   Maximum sequence length to precompute.
        device:              Target device.
    """

    def __init__(self, theta, d_k, max_seq_len, device=None):
        super().__init__()
        assert d_k % 2 == 0, "Head dimension d_k must be even for RoPE"

        # --- Precompute inverse frequencies ---
        # k = [0, 1, 2, ..., d_k/2 - 1]
        k = torch.arange(0, d_k // 2, dtype=torch.float32, device=device)

        # inv_freq[i] = 1 / theta^(2i / d_k)
        # Lower dimensions rotate fast (high frequency), higher dimensions rotate slow
        inv_freq = 1.0 / (theta ** (2 * k / d_k))

        # --- Precompute angles for all positions ---
        # positions = [0, 1, 2, ..., max_seq_len - 1]
        positions = torch.arange(max_seq_len, dtype=torch.float32, device=device)

        # angles[p, i] = p * inv_freq[i]  — shape: (max_seq_len, d_k/2)
        angles = torch.outer(positions, inv_freq)

        # Duplicate angles so each pair (i, i + d_k/2) gets the same angle
        # Final shape: (max_seq_len, d_k)
        angles = torch.cat([angles, angles], dim=-1)

        # Cache cos and sin tables — not model parameters, not saved in checkpoints
        self.register_buffer("cos_cache", angles.cos(), persistent=False)
        self.register_buffer("sin_cache", angles.sin(), persistent=False)

    def _rotate_half(self, x):
        """
        Rearrange x to represent the perpendicular rotation component.

        For a 2D vector [a, b], the perpendicular (90° rotation) is [-b, a].
        For a d_k-dimensional vector split into two halves [x1, x2]:
            rotate_half([x1, x2]) = [-x2, x1]

        Args:
            x: (..., d_k)
        Returns:
            Tensor of same shape: [-x[d_k/2:], x[:d_k/2]]
        """
        half = x.shape[-1] // 2
        # Second half (negated) goes first, then first half
        return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

    def forward(self, x, token_positions):
        """
        Apply RoPE rotation to x at the given token positions.

        x_rotated = x * cos(θ) + rotate_half(x) * sin(θ)

        This is the discrete analog of rotating a 2D vector:
            [a, b] → [a*cos - b*sin, a*sin + b*cos]

        Args:
            x:               (..., seq_len, d_k) — query or key tensor
            token_positions: (seq_len,) — integer positions to look up

        Returns:
            Rotated tensor of same shape as x.
        """
        # Look up precomputed cos and sin for each position
        cos = self.cos_cache[token_positions]  # (seq_len, d_k)
        sin = self.sin_cache[token_positions]  # (seq_len, d_k)

        # Apply rotation: x * cos + perpendicular(x) * sin
        return x * cos + self._rotate_half(x) * sin


## 8. Causal Multi-Head Self-Attention

Projects input into Q, K, V with learned weight matrices, splits into multiple heads, applies RoPE to Q and K, runs scaled dot-product attention with a causal (lower-triangular) mask, then merges heads and projects output.

In [9]:
class CausalMultiHeadSelfAttention(nn.Module):
    """
    Causal Multi-Head Self-Attention with Rotary Positional Embeddings.

    Each token attends to all previous tokens (including itself) but NOT
    to future tokens — enforced via a lower-triangular boolean mask.

    Heads run in parallel: d_model is split evenly across num_heads,
    each head has dimension d_k = d_model // num_heads.

    RoPE is applied to Q and K after splitting into heads.
    V is NOT rotated — position encodes where to look, not what to say.

    Args:
        d_model     (int):   Total model dimension.
        num_heads   (int):   Number of attention heads.
        max_seq_len (int):   Maximum sequence length (for RoPE precomputation).
        theta       (float): RoPE base frequency.
        device:              Target device.
        dtype:               Target dtype.
    """

    def __init__(self, d_model, num_heads, max_seq_len, theta=10000.0, device=None, dtype=None):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head

        # Four projection matrices: Q, K, V, and output
        # All project from/to d_model (heads are carved out after projection)
        self.W_Q = Linear(d_model, d_model, device=device, dtype=dtype)
        self.W_K = Linear(d_model, d_model, device=device, dtype=dtype)
        self.W_V = Linear(d_model, d_model, device=device, dtype=dtype)
        self.W_O = Linear(d_model, d_model, device=device, dtype=dtype)

        # RoPE module — precomputes cos/sin tables up to max_seq_len
        self.rope = RotaryPositionalEmbedding(
            theta=theta, d_k=self.d_k, max_seq_len=max_seq_len, device=device
        )

    def forward(self, x):
        """
        Run causal multi-head self-attention.

        Args:
            x: (batch_size, seq_len, d_model)

        Returns:
            Output tensor of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape

        # --- Step 1: Project input to Q, K, V ---
        # Each is shape: (batch_size, seq_len, d_model)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # --- Step 2: Split into heads ---
        # Reshape: (batch, seq, d_model) -> (batch, num_heads, seq, d_k)
        # Transpose puts heads before seq so attention runs per-head
        def split_heads(t):
            """Reshape (batch, seq, d_model) -> (batch, num_heads, seq, d_k)"""
            t = t.view(batch_size, seq_len, self.num_heads, self.d_k)
            return t.transpose(1, 2)  # -> (batch, num_heads, seq, d_k)

        Q = split_heads(Q)  # (batch, heads, seq, d_k)
        K = split_heads(K)  # (batch, heads, seq, d_k)
        V = split_heads(V)  # (batch, heads, seq, d_k)

        # --- Step 3: Apply RoPE to Q and K only ---
        # positions = [0, 1, 2, ..., seq_len - 1]
        positions = torch.arange(seq_len, device=x.device)
        Q = self.rope(Q, positions)  # Rotate queries
        K = self.rope(K, positions)  # Rotate keys
        # V is NOT rotated — values represent content, not position

        # --- Step 4: Build causal mask ---
        # Lower-triangular boolean matrix: True = allowed to attend
        # Position i can attend to positions 0..i (including itself)
        causal_mask = torch.tril(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device)
        )

        # --- Step 5: Scaled dot-product attention (parallel across all heads) ---
        # attn_out: (batch, num_heads, seq, d_k)
        attn_out = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

        # --- Step 6: Merge heads back ---
        # (batch, heads, seq, d_k) -> (batch, seq, heads, d_k) -> (batch, seq, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch_size, seq_len, self.d_model)

        # --- Step 7: Output projection ---
        return self.W_O(attn_out)


## 9. Transformer Block

One full block: **pre-norm attention** + residual, then **pre-norm FFN** + residual.

Pre-norm means the normalization happens *before* each sub-layer, not after. This is the modern convention (GPT-2 onward) — it makes training more stable because the residual stream stays un-normalized and easy to route gradients through.

In [10]:
class TransformerBlock(nn.Module):
    """
    A single Transformer block with pre-norm and residual connections.

    Architecture (pre-norm style):
        x = x + Attention( RMSNorm(x) )
        x = x + FFN( RMSNorm(x) )

    The residual stream x flows unchanged through both additions,
    giving gradients a direct highway back to early layers.

    Args:
        d_model     (int):   Model dimension.
        num_heads   (int):   Number of attention heads.
        d_ff        (int):   FFN hidden dimension.
        max_seq_len (int):   Max sequence length (for RoPE).
        theta       (float): RoPE base frequency.
        device:              Target device.
        dtype:               Target dtype.
    """

    def __init__(self, d_model, num_heads, d_ff, max_seq_len, theta=10000.0, device=None, dtype=None):
        super().__init__()

        # Pre-norm before attention
        self.norm1 = RMSNorm(d_model, device=device, dtype=dtype)

        # Causal multi-head self-attention with RoPE
        self.attn = CausalMultiHeadSelfAttention(
            d_model, num_heads, max_seq_len, theta, device, dtype
        )

        # Pre-norm before feed-forward
        self.norm2 = RMSNorm(d_model, device=device, dtype=dtype)

        # SwiGLU feed-forward network
        self.ffn = SwiGLUFFN(d_model=d_model, d_ff=d_ff, device=device, dtype=dtype)

    def forward(self, x):
        """
        Forward pass through one Transformer block.

        Args:
            x: (batch_size, seq_len, d_model)

        Returns:
            Tensor of same shape as x.
        """
        # Attention sub-layer with pre-norm and residual
        # norm1 normalizes x, attention refines it, residual adds it back
        x = x + self.attn(self.norm1(x))

        # FFN sub-layer with pre-norm and residual
        x = x + self.ffn(self.norm2(x))

        return x


## 10. Full Transformer Language Model

Stack N Transformer blocks after a token embedding layer. Apply a final RMSNorm, then a linear head that projects to vocabulary logits.

The output is **unnormalized logits** — apply `softmax` or `log_softmax` outside this module.

In [11]:
class TransformerLM(nn.Module):
    """
    Full Transformer Language Model.

    Architecture:
        token_ids
          -> Embedding lookup
          -> N x TransformerBlock
          -> RMSNorm
          -> Linear (lm_head) -> logits over vocab

    Returns unnormalized logits. Apply cross-entropy loss externally
    (PyTorch's nn.CrossEntropyLoss accepts logits directly).

    Args:
        vocab_size      (int):   Number of tokens in the vocabulary.
        context_length  (int):   Maximum sequence length.
        d_model         (int):   Model dimension.
        num_layers      (int):   Number of Transformer blocks.
        num_heads       (int):   Number of attention heads per block.
        d_ff            (int):   FFN hidden dimension. Auto-computed if None.
        theta           (float): RoPE base frequency.
        device:                  Target device.
        dtype:                   Target dtype.
    """

    def __init__(
        self,
        vocab_size,
        context_length,
        d_model,
        num_layers,
        num_heads,
        d_ff=None,
        theta=10000.0,
        device=None,
        dtype=None,
    ):
        super().__init__()

        if d_ff is None:
            # Default: parameter-equivalent to 4x FFN, rounded to nearest 64
            d_ff = math.ceil(int(8 / 3 * d_model) / 64) * 64

        # Token embedding table: vocab_size rows, d_model columns
        self.token_embedding = Embedding(vocab_size, d_model, device=device, dtype=dtype)

        # Stack of N identical Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, context_length, theta, device, dtype)
            for _ in range(num_layers)
        ])

        # Final normalization before the output projection
        self.final_norm = RMSNorm(d_model, device=device, dtype=dtype)

        # Output (language model) head: d_model -> vocab_size
        # Produces one logit per vocabulary token per position
        self.lm_head = Linear(d_model, vocab_size, device=device, dtype=dtype)

    def forward(self, token_ids):
        """
        Forward pass through the full Transformer LM.

        Args:
            token_ids: (batch_size, seq_len) integer tensor of token IDs

        Returns:
            logits: (batch_size, seq_len, vocab_size) — unnormalized log-probabilities
        """
        # Convert token IDs to dense vectors
        x = self.token_embedding(token_ids)  # (batch, seq, d_model)

        # Pass through every Transformer block sequentially
        for block in self.blocks:
            x = block(x)  # (batch, seq, d_model) — shape unchanged

        # Normalize the final hidden states
        x = self.final_norm(x)

        # Project to vocabulary logits
        return self.lm_head(x)  # (batch, seq, vocab_size)


## 11. Sanity Check — GPT-2 Small Config

Instantiate a GPT-2 small equivalent and run a forward pass on random token IDs.
Expected output shape: `(2, 128, 50257)`.

In [12]:
# GPT-2 Small configuration
VOCAB_SIZE = 50257      # GPT-2 vocabulary size
CONTEXT_LENGTH = 1024   # Maximum sequence length
D_MODEL = 768           # Model dimension
NUM_LAYERS = 12         # Number of Transformer blocks
NUM_HEADS = 12          # Number of attention heads
THETA = 10000.0         # RoPE base frequency

# Instantiate the model
model = TransformerLM(
    vocab_size=VOCAB_SIZE,
    context_length=CONTEXT_LENGTH,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    theta=THETA,
)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"  (~{total_params / 1e6:.1f}M — GPT-2 Small is ~117M)")

# --- Forward pass sanity check ---
# Random token IDs: batch of 2 sequences, each 128 tokens long
batch_size = 2
seq_len = 128
token_ids = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len))

# Run forward pass
with torch.no_grad():
    logits = model(token_ids)

# Expected: (2, 128, 50257)
print(f"\nInput shape:  {tuple(token_ids.shape)}")
print(f"Output shape: {tuple(logits.shape)}")
assert logits.shape == (batch_size, seq_len, VOCAB_SIZE), "Shape mismatch!"
print("\nAll good! Shape check passed.")


Total parameters: 162,148,608
  (~162.1M — GPT-2 Small is ~117M)

Input shape:  (2, 128)
Output shape: (2, 128, 50257)

All good! Shape check passed.
